<!-- ============================================================ -->
<!-- NOTEBOOK HEADER — MLOps Introductory Course on GCP           -->
<!-- ============================================================ -->

<div style="border-bottom: 3px solid #4285F4; padding-bottom: 12px; margin-bottom: 20px;">

<div style="display: flex; align-items: center; justify-content: space-between;">
  <div>
    <img src="https://www.isae-supaero.fr/wp-content/uploads/2025/03/logo.svg" width="180">
  </div>
  <div style="text-align: right;">
    <img src="https://user-images.githubusercontent.com/63151412/167391313-4683cc69-2bf6-4597-b767-5c18e2bbbfa0.png" width="180">
  </div>
</div>

# Lab 02c — Hyperparameter Tuning & Endpoint Testing

**Course:** MLOps Introductory Course on GCP · M2 Data Science · ISAE-SUPAERO  
**Lab created by:** Headmind Partners AI & Blockchain  
**Estimated duration:** ~1h00

</div>

## 📋 Lab Overview

Choosing the right hyperparameters is one of the most impactful steps in building an ML model. But tuning alone is not enough — an MLOps workflow requires you to **track experiments**, **compare results**, and ultimately **deploy the best model** so it can serve predictions. In this lab you will perform a **grid search** over hyperparameters for a neural network trained on **FashionMNIST**, upload the results to **Vertex AI TensorBoard**, identify the best configuration, and **deploy it behind a Vertex AI endpoint** for online prediction.

### Learning Objectives

1. Define a hyperparameter search space using the TensorBoard HParams plugin.
2. Build a Keras model parameterized by the search space and run a grid search.
3. Log each trial to TensorBoard and upload results to Vertex AI TensorBoard.
4. Analyze results in the HParams dashboard to identify the best configuration.
5. Retrain the best model and deploy it to a Vertex AI endpoint.
6. Send prediction requests to the endpoint and verify the results.

### Notebook Structure

| Section | Title | Focus |
|---------|-------|-------|
| 0 | Setup | Install dependencies, imports, GCP config |
| 1 | Data Preparation | Load and normalize FashionMNIST |
| 2 | Hyperparameter Space | Define search ranges with `hp.HParam` |
| 3 | Training & Logging | Build a parameterized model, run grid search, log to TensorBoard |
| 4 | Vertex AI TensorBoard | Create instance and upload logs |
| 5 | Analysis | Interpret results and identify best config |
| 6 | Deploy & Predict | Deploy the best model to an endpoint and test it |
| 7 | Cleanup | Delete all cloud resources |

### How to Read This Notebook

- **`# TODO`** — Code you need to write. Look for the `######` delimiters.
- **`✏️ Question`** — A conceptual question. Write your answer in the markdown cell below it.
- Cells **without** a TODO are provided — read them, run them, and make sure you understand them.
- Documentation links are provided in 📖 callouts whenever a new API is introduced.

---
## 0 · Setup

### 0.1 Install dependencies

In [ ]:
%pip install --upgrade --quiet google-cloud-aiplatform

### 0.2 Imports

In [ ]:
import numpy as np
import tensorflow as tf
from tensorboard.plugins.hparams import api as hp
from google.cloud import aiplatform

# Load the TensorBoard notebook extension
%load_ext tensorboard

# Clear any logs from previous runs
!rm -rf ./logs/

print(f"TensorFlow version: {tf.__version__}")

### 0.3 Configuration

In [ ]:
# ── Constants ──
PROJECT_ID = "your-project-id"   # @param {type:"string"}
LOCATION = "europe-west3"         # @param {type:"string"}

##############################  TODO  ##############################
# Set your_name to a unique lowercase identifier (e.g. first letter of first name + last name).
your_name = ...  # TODO
####################################################################

aiplatform.init(project=PROJECT_ID, location=LOCATION)
print(f"✅ Vertex AI initialized — project={PROJECT_ID}")

---
## 1 · Data Preparation

We use the [FashionMNIST](https://github.com/zalandoresearch/fashion-mnist) dataset — 70,000 grayscale images of 10 clothing categories. It is a drop-in replacement for MNIST and is often used for quick prototyping.

> 📖 **Docs:** [`tf.keras.datasets.fashion_mnist`](https://www.tensorflow.org/api_docs/python/tf/keras/datasets/fashion_mnist)

In [ ]:
fashion_mnist = tf.keras.datasets.fashion_mnist
(x_train, y_train), (x_test, y_test) = fashion_mnist.load_data()

CLASS_NAMES = [
    "T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
    "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot",
]

print(f"Train: {x_train.shape}  |  Test: {x_test.shape}")
print(f"Labels: {len(set(y_train))} classes")

In [ ]:
# Normalize the pixel values to [0, 1] by dividing by 255.
x_train, x_test = x_train / 255.0, x_test / 255.0

print(f"Pixel range: [{x_train.min()}, {x_train.max()}]")

---
## 2 · Hyperparameter Space

We define three hyperparameters to tune:

- **`num_units`** — number of neurons in the dense hidden layer.
- **`dropout`** — dropout rate for regularization.
- **`optimizer`** — which optimizer to use.

The TensorBoard HParams plugin provides `hp.Discrete` for categorical / discrete values and `hp.RealInterval` for continuous ranges.

> 📖 **Docs:** [TensorBoard HParams plugin](https://www.tensorflow.org/tensorboard/hyperparameter_tuning_with_hparams)

In [ ]:
##############################  TODO  ##############################
# Define three HParam objects:
# - HP_NUM_UNITS: a Discrete hparam named "num_units" with values 16 and 32
# - HP_DROPOUT: a RealInterval hparam named "dropout" ranging from 0.1 to 0.2
# - HP_OPTIMIZER: a Discrete hparam named "optimizer" with values "adam" and "sgd"

HP_NUM_UNITS = ...  # TODO
HP_DROPOUT = ...  # TODO
HP_OPTIMIZER = ...  # TODO
####################################################################

METRIC_NAME = "accuracy"

with tf.summary.create_file_writer("logs/hparam_tuning").as_default():
    hp.hparams_config(
        hparams=[HP_NUM_UNITS, HP_DROPOUT, HP_OPTIMIZER],
        metrics=[hp.Metric(METRIC_NAME, display_name=METRIC_NAME)],
    )

print("✅ Hyperparameter space configured.")

**✏️ Question 1 — Search space design**

a) How many total trials will a full grid search produce with the values defined above? Show the calculation.

b) What trade-off does each hyperparameter control?

---
*Your answer:*



---

---
## 3 · Training & Logging

We need two functions:
1. `train_test_model(hparams)` — builds, compiles, trains, and evaluates a Keras model given hyperparameters.
2. `run(run_dir, hparams)` — wraps the training in a TensorBoard logging context.

### 3.1 Training function

> 📖 **Docs:** [`tf.keras.layers`](https://www.tensorflow.org/api_docs/python/tf/keras/layers) [`tf.keras.Model`](https://www.tensorflow.org/api_docs/python/tf/keras/Model)

In [ ]:
##############################  TODO  ##############################
# Complete the train_test_model function.
# The model should be a Sequential with:
#   - a Flatten layer
#   - a Dense layer using hparams[HP_NUM_UNITS] units and relu activation
#   - a Dropout layer using hparams[HP_DROPOUT]
#   - a Dense output layer with 10 units and softmax activation
# Compile with the optimizer from hparams[HP_OPTIMIZER],
#   loss="sparse_categorical_crossentropy", and metrics=["accuracy"].
# Train for 1 epoch and return the test accuracy.

def train_test_model(hparams):
    model = tf.keras.models.Sequential([  # TODO
        ...
    ])
    model.compile(  # TODO
        ...
    )

    # Train for 1 epoch (fast demo — increase for real tuning)
    model.fit(x_train, y_train, epochs=1)
    _, accuracy = model.evaluate(x_test, y_test)
    return accuracy
####################################################################

### 3.2 Logging wrapper

This helper logs the hparams and the resulting metric for each trial to TensorBoard.

> 📖 **Key functions:**
> - [`hp.hparams()`](https://www.tensorflow.org/tensorboard/hyperparameter_tuning_with_hparams) — records hyperparameter values for a run.
> - [`tf.summary.scalar()`](https://www.tensorflow.org/api_docs/python/tf/summary/scalar) — logs a scalar metric value.

In [ ]:
def run(run_dir, hparams):
    """Run a single trial: train the model and log results to TensorBoard."""
    with tf.summary.create_file_writer(run_dir).as_default():
        hp.hparams(hparams)  # record the values used in this trial
        accuracy = train_test_model(hparams)
        tf.summary.scalar(METRIC_NAME, accuracy, step=1)

---
## 4 · Grid Search

We iterate over all combinations of hyperparameters. For continuous ranges (dropout), we sample only the min and max to keep the total number of trials manageable.

With 2 `num_units` × 2 `dropout` × 2 `optimizer` = **8 trials**, each training for 1 epoch, this should take a few minutes.

In [ ]:
session_num = 0

for num_units in HP_NUM_UNITS.domain.values:
    for dropout_rate in (HP_DROPOUT.domain.min_value, HP_DROPOUT.domain.max_value):
        for optimizer in HP_OPTIMIZER.domain.values:
            hparams = {
                HP_NUM_UNITS: num_units,
                HP_DROPOUT: dropout_rate,
                HP_OPTIMIZER: optimizer,
            }
            run_name = f"run-{session_num}"
            print(f"--- Starting trial: {run_name}")
            print({h.name: hparams[h] for h in hparams})
            run(f"logs/hparam_tuning/{run_name}", hparams)
            session_num += 1

print(f"\n✅ Grid search complete — {session_num} trials logged.")

---
## 5 · Vertex AI TensorBoard

**Vertex AI TensorBoard** is a managed version of the open-source TensorBoard. It adds persistent, shareable dashboards, project-level experiment listing, and integration with other Vertex AI services.

> 📖 **Docs:** [Vertex AI TensorBoard overview](https://cloud.google.com/vertex-ai/docs/experiments/tensorboard-introduction)

### 5.1 Create a TensorBoard instance

> 📖 **Docs:** [`aiplatform.Tensorboard.create()`](https://docs.cloud.google.com/python/docs/reference/aiplatform/latest/google.cloud.aiplatform.Tensorboard#google_cloud_aiplatform_Tensorboard_create)

In [ ]:
##############################  TODO  ##############################
# Create a Vertex AI TensorBoard instance using aiplatform.Tensorboard.create().
# Use TENSORBOARD_NAME as the display_name.
# Don't forget to add the project id and location.

TENSORBOARD_NAME = f"lab02c-tb-{your_name}"

tensorboard = ...  # TODO
####################################################################

TENSORBOARD_RESOURCE_NAME = tensorboard.gca_resource.name
print(f"✅ TensorBoard resource name: {TENSORBOARD_RESOURCE_NAME}")

### 5.2 Upload logs

The `tb-gcp-uploader` CLI tool pushes local TensorBoard logs to the Vertex AI instance.

> 📖 **Docs:** [Upload TensorBoard logs](https://cloud.google.com/vertex-ai/docs/experiments/tensorboard-setup#upload-logs)

In [ ]:
EXPERIMENT_NAME = f"lab02c-hparam-tuning-{your_name}"

!tb-gcp-uploader --one_shot=True \
    --tensorboard_resource_name=$TENSORBOARD_RESOURCE_NAME \
    --logdir="logs/hparam_tuning/" \
    --experiment_name=$EXPERIMENT_NAME

> 💡 **Tip:** Click the link generated above (or navigate to the Vertex AI console → TensorBoard) to open the dashboard.

**✏️ Question 2 — Managed TensorBoard**

a) What advantages does Vertex AI TensorBoard offer compared to running TensorBoard locally on your machine?

b) In Lab 02 you used `Vertex AI Experiments` to track runs with `log_params()` and `log_metrics()`. How does that compare to the TensorBoard HParams approach used here? When would you prefer one over the other?

---
*Your answer:*



---

---
## 6 · Analysis

The HParams dashboard provides three interactive views:

- **Table View** — lists all runs with their hyperparameters and metrics.
- **Parallel Coordinates View** — shows how hyperparameter values relate to the metric.
- **Scatter Plot Matrix View** — pairwise scatter plots of hparams vs. metric.

**✏️ Question 3 — Best configuration**

a) Which hyperparameter combination achieved the highest accuracy?

b) What would you change to get a more reliable comparison?

---
*Your answer:*



---

---
## 7 · Deploy & Predict

Now that you have identified the best hyperparameters, the next step in an MLOps workflow is to **retrain the best model**, **save it**, and **deploy it behind an endpoint** so it can serve live predictions.

### 7.1 Retrain the best model

We retrain with the best hyperparameters found during the grid search, but this time for **5 epochs** to get a more reliable model.

In [ ]:
##############################  TODO  ##############################
# Define best_hparams as a dictionary with the best hyperparameters
# from your grid search analysis.
# Keys should be HP_NUM_UNITS, HP_DROPOUT, HP_OPTIMIZER.

best_hparams = ...  # TODO
####################################################################

print("Best hyperparameters:")
print({h.name: best_hparams[h] for h in best_hparams})

In [ ]:
# Retrain with best hparams for 5 epochs
best_model = tf.keras.models.Sequential([
    tf.keras.layers.Flatten(input_shape=(28, 28)),
    tf.keras.layers.Dense(best_hparams[HP_NUM_UNITS], activation="relu"),
    tf.keras.layers.Dropout(best_hparams[HP_DROPOUT]),
    tf.keras.layers.Dense(10, activation="softmax"),
])
best_model.compile(
    optimizer=best_hparams[HP_OPTIMIZER],
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

best_model.fit(x_train, y_train, epochs=5, validation_split=0.1)
_, test_acc = best_model.evaluate(x_test, y_test)
print(f"\n✅ Best model test accuracy: {test_acc:.4f}")

### 7.2 Save the model to Cloud Storage

Vertex AI endpoints serve models from **Cloud Storage artifacts**. We save the Keras model in TensorFlow's SavedModel format.

> 📖 **Docs:** [Vertex AI pre-built containers](https://cloud.google.com/vertex-ai/docs/training/pre-built-containers)

In [ ]:
BUCKET_URI = "gs://your-bucket-name"  # @param {type:"string"}
MODEL_DIR = f"{BUCKET_URI}/{your_name}/lab02c-best-model"

best_model.export(MODEL_DIR)
print(f"✅ Model saved to {MODEL_DIR}")

### 7.3 Upload to Model Registry & deploy

We upload the saved model to the Vertex AI Model Registry, create an endpoint, and deploy the model to it.

> 📖 **Key functions:**
> - [`aiplatform.Model.upload()`](https://cloud.google.com/python/docs/reference/aiplatform/latest/google.cloud.aiplatform.Model#google_cloud_aiplatform_Model_upload)
> - [`aiplatform.Endpoint.create()`](https://cloud.google.com/python/docs/reference/aiplatform/latest/google.cloud.aiplatform.Endpoint#google_cloud_aiplatform_Endpoint_create)
> - [`endpoint.deploy()`](https://cloud.google.com/python/docs/reference/aiplatform/latest/google.cloud.aiplatform.Endpoint#google_cloud_aiplatform_Endpoint_deploy)

In [ ]:
##############################  TODO  ##############################
# Upload the saved model to the Vertex AI Model Registry.
# Use aiplatform.Model.upload() with:
#   - display_name: a descriptive name including your_name
#   - artifact_uri: MODEL_DIR
#   - serving_container_image_uri: the pre-built TF container below

DEPLOY_IMAGE = "us-docker.pkg.dev/vertex-ai/prediction/tf2-cpu.2-15:latest"

model = ...  # TODO
####################################################################

print(f"✅ Model uploaded: {model.display_name} (resource: {model.resource_name})")

In [ ]:
##############################  TODO  ##############################
# Create an endpoint and deploy the model to it.
# 1. Create an endpoint with aiplatform.Endpoint.create()
# 2. Deploy the model to the endpoint with endpoint.deploy()
#    Use machine_type="n1-standard-4"
ENDPOINT_DISPLAY_NAME = f"lab02c-endpoint-{your_name}"

endpoint = ...  # TODO

endpoint.deploy(  # TODO
    ...
)
####################################################################

print(f"✅ Model deployed to endpoint: {endpoint.resource_name}")

> ⏳ **Deployment takes 5–10 minutes.** While you wait, go back to the Vertex AI TensorBoard dashboard and explore the HParams views.

### 7.4 Send predictions

Once the endpoint is ready, we can send prediction requests. The FashionMNIST model expects a flattened 28×28 image as input (a list of 784 floats). The endpoint returns a list of 10 class probabilities.

In [ ]:
##############################  TODO  ##############################
# Send a prediction for the first 3 test images.
# 1. Prepare the instances as a list of lists (each image as a list of floats)
# 2. Call endpoint.predict(instances=...)
# 3. Print the predicted class names and the true labels

test_images = x_test[:3]
instances = ...  # TODO
response = ...  # TODO
####################################################################

print("Predictions:")
for i, prediction in enumerate(response.predictions):
    predicted_class = np.argmax(prediction)
    true_class = y_test[i]
    print(
        f"  Image {i}: predicted={CLASS_NAMES[predicted_class]}, "
        f"true={CLASS_NAMES[true_class]} "
        f"{'✅' if predicted_class == true_class else '❌'}"
    )

**✏️ Question 4 — From tuning to production**

a) You used the test set during the grid search (in `train_test_model`) and then again to evaluate the retrained model. Why is this problematic? How would you fix this in a production setting?

b) In a real MLOps pipeline, what additional steps would you add between finding the best hyperparameters and deploying to production?

---
*Your answer:*



---

---
## 8 · Cleanup

Always clean up cloud resources when you are done to avoid unnecessary charges. The order matters: undeploy the model first, then delete the endpoint, then delete the model and TensorBoard instance.

In [ ]:
# Undeploy the model
deployed_model_id = endpoint.gca_resource.deployed_models[0].id
endpoint.undeploy(deployed_model_id)
print(f"✅ Model undeployed (ID: {deployed_model_id})")

In [ ]:
# Delete the endpoint
endpoint.delete()
print("✅ Endpoint deleted.")

In [ ]:
# Delete the model from the registry
model.delete()
print("✅ Model deleted from registry.")

In [ ]:
# Delete the TensorBoard instance
tensorboard.delete()
print("✅ TensorBoard instance deleted.")

---
## Summary

In this lab, you learned to:

| Step | What you did | Tool / Feature used |
|------|-------------|---------------------|
| Data | Loaded and normalized FashionMNIST | `tf.keras.datasets` |
| Search Space | Defined discrete and continuous hparam ranges | `hp.HParam`, `hp.Discrete`, `hp.RealInterval` |
| Training | Built a parameterized Keras model and logged trials | `tf.keras.Sequential`, `tf.summary`, `hp.hparams()` |
| Upload | Created a Vertex AI TensorBoard and uploaded logs | `aiplatform.Tensorboard.create()`, `tb-gcp-uploader` |
| Analysis | Compared runs in the HParams dashboard | Table / Parallel Coordinates / Scatter views |
| Deploy | Uploaded best model to registry and deployed to endpoint | `Model.upload()`, `Endpoint.create()`, `endpoint.deploy()` |
| Predict | Sent prediction requests and verified results | `endpoint.predict()` |
| Cleanup | Deleted all cloud resources | `undeploy()`, `delete()` |

**Next lab:** You will build **batch and streaming inference endpoints** and explore how to serve predictions at scale.